# 13 Validator

Independent check of the final MCAP files against the Vendor Deliverable Standard.

## 1 Setup

Imports, flags, paths and the list of episodes. It also prints the size and sha256 of each final file.

In [1]:
import json, hashlib, shutil, subprocess, sys, re, random, zipfile, traceback, datetime as dt
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import yaml
import cv2
import av
import jsonschema
from tqdm.auto import tqdm
from mcap.reader import make_reader
from foxglove_schemas_protobuf.CompressedVideo_pb2 import CompressedVideo

# flags
FORCE = False
FULL_DECODE = False
SAMPLES_POSE_PER_EPISODE = 30
SAMPLES_LABELS = 30
SEED = 13
TARGET_MIN_S = 120
TARGET_MAX_S = 600
HARD_MAX_S = 1200
COVERAGE_FLOOR_PCT = 30
BITRATE_REF = (5, 8)
PLACEHOLDERS = ["vendorX", "op01", "site-unknown"]
MCAP_EXE = None

# paths
PROJECT_ROOT = Path.cwd().parent
CONFIG = PROJECT_ROOT / "configs" / "config.yaml"
if not CONFIG.exists():
    raise SystemExit(f"missing {CONFIG}: the working directory must be a folder inside the project (project root = its parent)")
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
REPORTS = PROJECT_ROOT / "reports"
FINAL = PROJECT_ROOT / "mcap" / "final"
INTERIM = PROJECT_ROOT / "interim" / "validate"
REPORTS.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

# names from the standard
EXPECTED_CAMS = ["cam_high", "cam_left_wrist", "cam_right_wrist", "cam_ext"]
BODY_CAMS = EXPECTED_CAMS[:3]
VIDEO_TOPICS = ["/cameras/" + c for c in EXPECTED_CAMS]
IMU_TOPICS = ["/ego/imu/" + c for c in BODY_CAMS]
HAND_TOPICS = ["/ego/hand/joints/left/position", "/ego/hand/joints/right/position"]
CALIB_TOPICS = ["/ego/calibration/intrinsics", "/ego/calibration/extrinsics"]
STANDARD_TOPICS = VIDEO_TOPICS + HAND_TOPICS
ALL_TOPICS = VIDEO_TOPICS + IMU_TOPICS + HAND_TOPICS + CALIB_TOPICS
REQUIRED_KEYS = ["episode_id", "capture_start_iso", "capture_stop_iso", "fps", "duration_sec",
                 "max_timesteps", "rig", "cameras", "user_id", "task_id", "task_name",
                 "language_instruction", "spans", "pose_modality", "coverage", "episode_tags",
                 "source_provenance"]

# results table, one entry per test id
HYP = {}
def add(tid, clause, hyp, measured, result, fix=""):
    measured = str(measured)
    if result == "FLAG":
        measured += " | fix: " + (fix or "see measured")
    HYP[tid] = dict(id=tid, clause=clause, hypothesis=hyp, measured=measured, result=result)
    print(f"{tid:5s} {result:5s} {hyp[:62]} | {measured[:120]}")

def show(df, n=50):
    print(df.head(n).to_string(index=False))
    if len(df) > n:
        print(f"... {len(df) - n} more rows not shown")

POSE_REVIEW = None
LABEL_REVIEW = None

p06 = REPORTS / "06_summary.json"
if not p06.exists():
    raise SystemExit(f"missing {p06}")
EPISODES = list(json.loads(p06.read_text(encoding="utf-8"))["episode_ids"])
PATHS = {e: FINAL / f"{e}.mcap" for e in EPISODES}
for e in [e for e in EPISODES if not PATHS[e].exists()]:
    print("missing final file:", PATHS[e])
EPISODES = [e for e in EPISODES if PATHS[e].exists()]
if not EPISODES:
    raise SystemExit("no final mcap files found")
print(len(EPISODES), "episodes")

c:\workstation\3RD YEAR\ego_pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4 episodes


In [2]:
def sha256_of(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

SHA = {}
for e in EPISODES:
    SHA[e] = sha256_of(PATHS[e])
    print(e, f"{PATHS[e].stat().st_size / 1e6:.1f} MB")
    print("  sha256", SHA[e])

# compare with the manifest of step 12
man_path = REPORTS / "12_mcap_manifest.csv"
if man_path.exists():
    man = pd.read_csv(man_path, dtype=str).fillna("")
    for e in EPISODES:
        hit = man[man.apply(lambda r: any(e in str(v) for v in r.values), axis=1)]
        if hit.empty:
            print("INFO", e, "not found in 12_mcap_manifest.csv")
            continue
        same = any(str(v).strip().lower() == SHA[e] for v in hit.iloc[0].values)
        print("manifest:", "same sha256" if same else "INFO different sha256", e)
else:
    print("INFO no 12_mcap_manifest.csv to compare with")

vendorX_2026-10-11_op01_000001 2354.3 MB
  sha256 24ecce0b0bdb294273dd38e11e06df18aa91228e1e7c7fd603e04675f99e46fa
vendorX_2026-10-11_op01_000002 976.2 MB
  sha256 a215245bfdfb09198f02022ba15301e0f109c8c2c97952af540361771ee81060
vendorX_2026-10-11_op01_000003 1116.6 MB
  sha256 ad0bd7cec4e9015116f31f23f304cbed97ccc6f6801807e13d64ff5a59156d48
vendorX_2026-10-11_op01_000004 1932.9 MB
  sha256 8f77387703bedbad2791016e9b2030d0347cc563b048fd9142655f03263ebb74
manifest: same sha256 vendorX_2026-10-11_op01_000001
manifest: same sha256 vendorX_2026-10-11_op01_000002
manifest: same sha256 vendorX_2026-10-11_op01_000003
manifest: same sha256 vendorX_2026-10-11_op01_000004


## 2 Streaming pass

One pass over each file. It collects the statistics of every channel, validates the JSON messages and writes the video streams to interim\validate.

In [3]:
class BadConst(Exception):
    pass

def bad_const(x):
    raise BadConst(x)

NAL_RE = re.compile(b"\x00\x00\x01")

def nal_types(data):
    out = []
    for m in NAL_RE.finditer(data):
        p = m.end()
        if p < len(data):
            out.append(data[p] & 31)
    return out

def parse_hand(h, obj):
    joints = obj.get("joints") or []
    names = [j.get("name") if isinstance(j, dict) else None for j in joints]
    if len(joints) != 25:
        h["len_bad"] += 1
    if h["names"] is None:
        h["names"] = names
    elif names != h["names"]:
        h["names_bad"] += 1
    xs, ys, cs = np.full(25, np.nan), np.full(25, np.nan), np.full(25, np.nan)
    try:
        for k, j in enumerate(joints[:25]):
            pos = j.get("position")
            if isinstance(pos, dict) and pos.get("x") is not None and pos.get("y") is not None:
                xs[k], ys[k] = float(pos["x"]), float(pos["y"])
            cf = j.get("confidence")
            if cf is not None:
                cs[k] = float(cf)
            if j.get("orientation") is not None:
                h["orient"] += 1
    except Exception:
        h["parse_err"] += 1
    h["x"].append(xs); h["y"].append(ys); h["c"].append(cs)
    h["det"].append(bool(obj.get("detected")))
    h["trk"].append(bool(obj.get("tracked")))
    h["oof"].append(bool(obj.get("out_of_frame")))
    h["idx"].append(obj.get("sample_index"))
    h["tag"][obj.get("position_tag")] += 1
    h["hand"][obj.get("hand")] += 1
    h["frame_id"][obj.get("frame_id")] += 1

def scan_file(e):
    chan, hands, calib, meta_recs, valid, fhs = {}, {}, {}, [], {}, {}
    prev, run = {}, {}
    cvs = CompressedVideo()
    with open(PATHS[e], "rb") as f:
        reader = make_reader(f)
        total = None
        try:
            total = reader.get_summary().statistics.message_count
        except Exception:
            pass
        try:
            for rec in reader.iter_metadata():
                meta_recs.append((rec.name, dict(rec.metadata)))
        except Exception as ex:
            print("  cannot read metadata records:", ex)
        for schema, channel, msg in tqdm(reader.iter_messages(), total=total, desc=e[-14:], unit="msg", mininterval=1):
            top = channel.topic
            st = chan.get(top)
            if st is None:
                st = dict(topic=top, schema_name=getattr(schema, "name", None), schema_enc=getattr(schema, "encoding", None),
                          log=[], bytes=0, pub_eq=0, emb_eq=0, emb_bad=0, emb_missing=0, invalid=0, nonfinite=0, first_error=None)
                if top.startswith("/cameras/"):
                    st.update(kind="video", au=[], idr=[], no_aud=0, idr_no_ps=0, dup=0, dup_run=0, frame_id=Counter(), fmt=Counter())
                    st["stream_path"] = INTERIM / f"{e}_{top.split('/')[-1]}.h264"
                    fhs[top] = open(st["stream_path"], "wb")
                else:
                    st["kind"] = "json"
                    v = None
                    if schema is not None and schema.encoding == "jsonschema" and schema.data:
                        sch = json.loads(schema.data)
                        v = jsonschema.validators.validator_for(sch)(sch)
                    valid[top] = v
                chan[top] = st
            i = len(st["log"])
            st["log"].append(msg.log_time)
            st["bytes"] += len(msg.data)
            if msg.publish_time == msg.log_time:
                st["pub_eq"] += 1
            if st["kind"] == "video":
                cvs.ParseFromString(msg.data)
                emb = cvs.timestamp.seconds * 1_000_000_000 + cvs.timestamp.nanos
                if emb == msg.log_time:
                    st["emb_eq"] += 1
                else:
                    st["emb_bad"] += 1
                st["frame_id"][cvs.frame_id] += 1
                st["fmt"][cvs.format] += 1
                au = bytes(cvs.data)
                fhs[top].write(au)
                st["au"].append(len(au))
                t = nal_types(au)
                if not t or t[0] != 9:
                    st["no_aud"] += 1
                if 5 in t:
                    st["idr"].append(i)
                    before = t[:t.index(5)]
                    if 7 not in before or 8 not in before:
                        st["idr_no_ps"] += 1
                if top in prev and au == prev[top]:
                    run[top] = run.get(top, 0) + 1
                    st["dup"] += 1
                    st["dup_run"] = max(st["dup_run"], run[top])
                else:
                    run[top] = 0
                prev[top] = au
                continue
            # json channels
            try:
                obj = json.loads(msg.data, parse_constant=bad_const)
            except BadConst:
                st["nonfinite"] += 1
                obj = json.loads(msg.data)
            except Exception as ex:
                st["invalid"] += 1
                st["first_error"] = st["first_error"] or ("not json: " + str(ex)[:100])
                continue
            ts = obj.get("timestamp") if isinstance(obj, dict) else None
            try:
                emb = int(ts["sec"]) * 1_000_000_000 + int(ts["nsec"])
                if emb == msg.log_time:
                    st["emb_eq"] += 1
                else:
                    st["emb_bad"] += 1
            except Exception:
                st["emb_missing"] += 1
            v = valid.get(top)
            if v is not None:
                err = next(iter(v.iter_errors(obj)), None)
                if err is not None:
                    st["invalid"] += 1
                    st["first_error"] = st["first_error"] or str(err.message)[:150]
            if top.startswith("/ego/hand/joints/") and isinstance(obj, dict):
                h = hands.get(top)
                if h is None:
                    h = hands[top] = dict(x=[], y=[], c=[], det=[], trk=[], oof=[], idx=[], names=None, names_bad=0, len_bad=0,
                                          orient=0, parse_err=0, tag=Counter(), hand=Counter(), frame_id=Counter())
                parse_hand(h, obj)
            elif top.startswith("/ego/calibration/"):
                calib.setdefault(top, []).append(obj)
    for fh in fhs.values():
        fh.close()
    # finish the statistics
    for top, st in chan.items():
        lg = np.array(st["log"], dtype=np.int64)
        st["log"], st["n"] = lg, len(lg)
        st["first"], st["last"] = int(lg[0]), int(lg[-1])
        d = np.diff(lg)
        if len(d):
            st.update(int_min=int(d.min()), int_med=float(np.median(d)), int_max=int(d.max()), n_equal=int((d == 0).sum()),
                      n_dec=int((d < 0).sum()), n_50=int((d >= 50_000_000).sum()), n_100=int((d >= 100_000_000).sum()))
        else:
            st.update(int_min=0, int_med=0.0, int_max=0, n_equal=0, n_dec=0, n_50=0, n_100=0)
        if st["kind"] == "video":
            st["au"] = np.array(st["au"], dtype=np.int64)
            st["idr"] = np.array(st["idr"], dtype=np.int64)
            st["offsets"] = np.concatenate([[0], np.cumsum(st["au"])])
    for h in hands.values():
        for k, new in (("x", "X"), ("y", "Y"), ("c", "C")):
            h[new] = np.array(h.pop(k))
        for k in ("det", "trk", "oof"):
            h[k] = np.array(h[k], dtype=bool)
    meta = next((m for n, m in meta_recs if n == "episode_metadata"), {})
    return dict(chan=chan, hands=hands, calib=calib, meta=meta, meta_names=[n for n, _ in meta_recs])

E = {}
for e in list(EPISODES):
    try:
        E[e] = scan_file(e)
        ch = E[e]["chan"]
        print(f"  {e}: {len(ch)} channels, {sum(s['n'] for s in ch.values())} messages")
    except Exception:
        print("scan failed for", e)
        traceback.print_exc()
        EPISODES.remove(e)

11_op01_000001: 100%|██████████| 191187/191187 [02:22<00:00, 1338.47msg/s]


  vendorX_2026-10-11_op01_000001: 11 channels, 191187 messages


11_op01_000002: 100%|██████████| 84525/84525 [01:02<00:00, 1343.67msg/s]


  vendorX_2026-10-11_op01_000002: 11 channels, 84525 messages


11_op01_000003: 100%|██████████| 93703/93703 [01:07<00:00, 1396.78msg/s]


  vendorX_2026-10-11_op01_000003: 11 channels, 93703 messages


11_op01_000004: 100%|██████████| 162022/162022 [01:59<00:00, 1352.84msg/s]


  vendorX_2026-10-11_op01_000004: 11 channels, 162022 messages


## 3 Episodes and format

Sections 1 and 2 of the standard: episode length, time windows, gaps, clock, metadata record, topic names and embedded timestamps.

In [4]:
def tofloat(v):
    try:
        return float(v)
    except Exception:
        return float("nan")

def iso_ns(s):
    d = dt.datetime.fromisoformat(str(s).replace("Z", "+00:00"))
    if d.tzinfo is None:
        d = d.replace(tzinfo=dt.timezone.utc)
    return (d - dt.datetime(1970, 1, 1, tzinfo=dt.timezone.utc)) // dt.timedelta(microseconds=1) * 1000

# parsed metadata
JSONISH = ["fps", "duration_sec", "max_timesteps", "rig", "cameras", "spans", "coverage", "episode_tags", "source_provenance"]
PM = {}
for e in EPISODES:
    m = E[e]["meta"]
    p = dict(m)
    for k in JSONISH:
        if k in m:
            try:
                p[k] = json.loads(m[k])
            except Exception:
                p[k] = m[k]
    PM[e] = p
DUR = {e: tofloat(E[e]["meta"].get("duration_sec")) for e in EPISODES}
START, STOP = {}, {}
for e in EPISODES:
    for dct, k in ((START, "capture_start_iso"), (STOP, "capture_stop_iso")):
        try:
            dct[e] = iso_ns(E[e]["meta"][k])
        except Exception:
            dct[e] = None

# S1.1 duration
over = [e for e in EPISODES if DUR[e] > HARD_MAX_S]
outside = [f"{e[-12:]} {DUR[e]:.0f}s" for e in EPISODES if not (TARGET_MIN_S <= DUR[e] <= TARGET_MAX_S)]
res = "FLAG" if over else ("INFO" if outside else "PASS")
add("S1.1", "1", f"duration at most {HARD_MAX_S}s, target {TARGET_MIN_S}-{TARGET_MAX_S}s",
    "durations " + ", ".join(f"{DUR[e]:.1f}s" for e in EPISODES) + ("; outside target: " + ", ".join(outside) if outside else ""),
    res, "cut or split the episodes above the hard cap")

# S1.2 no overlap in utc
span = sorted((START[e], STOP[e], e) for e in EPISODES if START[e] and STOP[e])
ov = [(a[2][-12:], b[2][-12:]) for a, b in zip(span, span[1:]) if b[0] < a[1]]
add("S1.2", "1", "episodes do not overlap in UTC time", f"{len(span)} episodes checked, overlaps: {ov or 'none'}",
    "FLAG" if ov or len(span) < len(EPISODES) else "PASS", "check capture_start_iso and capture_stop_iso in the metadata")

# channel table
ch_rows = []
for e in EPISODES:
    for top, st in E[e]["chan"].items():
        s0, s1 = START[e], STOP[e]
        ch_rows.append(dict(ep=e[-12:], topic=top.replace("/ego/", "").replace("/cameras/", "cam:"), n=st["n"],
                            first_ms=(st["first"] - s0) / 1e6 if s0 else np.nan, last_ms=(st["last"] - s1) / 1e6 if s1 else np.nan,
                            min_ms=st["int_min"] / 1e6, med_ms=st["int_med"] / 1e6, max_ms=st["int_max"] / 1e6,
                            equal=st["n_equal"], dec=st["n_dec"], ge50=st["n_50"], ge100=st["n_100"], _e=e, _top=top))
CH = pd.DataFrame(ch_rows)
show(CH.drop(columns=["_e", "_top"]).round(1))

# S1.3 channels inside capture window
is_cal = CH._top.str.startswith("/ego/calibration/")
# calibration holds one message, so it only has to lie inside the window
bad = CH[(~is_cal & ((CH.first_ms.abs() > 100) | (CH.last_ms.abs() > 100))) | (is_cal & ((CH.first_ms < -100) | (CH.last_ms > 100))) | CH.first_ms.isna()]
name_ok = [e for e in EPISODES if E[e]["meta"].get("episode_id") == e]
add("S1.3", "1", "one file per episode, every channel within 100 ms of the capture window",
    f"{len(EPISODES)} files, episode_id matches file name in {len(name_ok)}, channels outside window: {len(bad)}" +
    (" e.g. " + ", ".join(f"{r.ep}/{r.topic} {r.first_ms:.0f}/{r.last_ms:.0f} ms" for r in bad.head(3).itertuples()) if len(bad) else ""),
    "FLAG" if len(bad) or len(name_ok) < len(EPISODES) else "PASS", "re-export so all channels share the capture window; episode_id must equal the file name")

# S2.1 gaps
vh = CH[CH._top.str.startswith("/cameras/") | CH._top.str.contains("/hand/")]
big = vh[vh.ge100 > 0]
add("S2.1", "2", "no gaps of 100 ms or more in video and hand channels",
    f"largest gap video/hand {vh.max_ms.max():.1f} ms, channels with gaps >= 100 ms: {len(big)}; all channels {int(CH.ge100.sum())} gaps",
    "FLAG" if len(big) else "PASS", "find the gaps above 100 ms in " + ", ".join(sorted(set(big.topic))) if len(big) else "")

# S2.2 monotonic clock
dec, eq = int(CH.dec.sum()), int(CH.equal.sum())
add("S2.2", "2", "log times are non-decreasing and strictly increasing per channel",
    f"decreasing intervals {dec}, equal-to-previous intervals {eq}, channels with equal intervals {int((CH.equal > 0).sum())}",
    "FLAG" if dec else ("INFO" if eq else "PASS"), "log times must never go backwards")

# S2.3 metadata record
probs = []
for e in EPISODES:
    m = E[e]["meta"]
    n_rec = E[e]["meta_names"].count("episode_metadata")
    miss, extra = [k for k in REQUIRED_KEYS if k not in m], [k for k in m if k not in REQUIRED_KEYS]
    nonstr = [k for k, v in m.items() if not isinstance(v, str)]
    if n_rec != 1 or len(m) != 17 or miss or extra or nonstr:
        probs.append(f"{e[-12:]}: records {n_rec}, keys {len(m)}, missing {miss}, extra {extra}, non-string {nonstr}")
add("S2.3", "2", "one native record named episode_metadata with 17 string values",
    f"{len(EPISODES) - len(probs)} of {len(EPISODES)} ok" + ("; " + "; ".join(probs) if probs else ""), "FLAG" if probs else "PASS",
    "write exactly one metadata record episode_metadata with the 17 keys")

# S2.4 topics
tp = []
for e in EPISODES:
    have = set(E[e]["chan"])
    if set(STANDARD_TOPICS) - have:
        tp.append(f"{e[-12:]} missing {sorted(set(STANDARD_TOPICS) - have)}")
add("S2.4", "2", "standard topic names for cameras and hand joints", "all present" if not tp else "; ".join(tp), "FLAG" if tp else "PASS",
    "use the exact standard topic names")

# S2.5 camera names
cp = []
for e in EPISODES:
    for c in EXPECTED_CAMS:
        st = E[e]["chan"].get("/cameras/" + c)
        if st is None:
            continue
        if set(st["frame_id"]) != {c} or set(st["fmt"]) != {"h264"}:
            cp.append(f"{e[-12:]}/{c} frame_id {dict(st['frame_id'])} format {dict(st['fmt'])}")
add("S2.5", "2", "camera names exact (frame_id = camera name, format h264)", "ok" if not cp else "; ".join(cp[:3]), "FLAG" if cp else "PASS",
    "frame_id must equal the camera name and format must be h264")

# S2.6 extra topics
extra = sorted({t for e in EPISODES for t in E[e]["chan"]} - set(STANDARD_TOPICS))
add("S2.6", "2", "extra non-standard topics are listed", f"{len(extra)} extra: {extra}", "INFO" if extra else "PASS")

# S2.7 embedded timestamps and publish time
tot = sum(st["n"] for e in EPISODES for st in E[e]["chan"].values())
emb_bad = sum(st["emb_bad"] + st["emb_missing"] for e in EPISODES for st in E[e]["chan"].values())
pub_bad = sum(st["n"] - st["pub_eq"] for e in EPISODES for st in E[e]["chan"].values())
add("S2.7", "2", "embedded timestamp equals log_time for every message (publish_time equals log_time)",
    f"{tot} messages, embedded timestamp differs or missing {emb_bad}, publish_time differs {pub_bad}",
    "FLAG" if emb_bad else ("INFO" if pub_bad else "PASS"), "rewrite the timestamp field of the channels with mismatches")

# S2.8 json schemas
jb = []
for e in EPISODES:
    for top, st in E[e]["chan"].items():
        if st["kind"] == "json" and (st["invalid"] or st["nonfinite"] or st["schema_enc"] != "jsonschema"):
            jb.append(f"{e[-12:]}{top} invalid {st['invalid']} nonfinite {st['nonfinite']} schema {st['schema_enc']} {st['first_error']}")
add("S2.8", "2", "JSON messages valid against the embedded schema, no NaN or Infinity", "all valid" if not jb else "; ".join(jb[:3]),
    "FLAG" if jb else "PASS", "fix the messages or the embedded schema of the channels listed")

S1.1  INFO  duration at most 1200s, target 120-600s | durations 708.0s, 313.0s, 347.0s, 600.0s; outside target: _op01_000001 708s
S1.2  PASS  episodes do not overlap in UTC time | 4 episodes checked, overlaps: none
          ep                      topic     n  first_ms   last_ms  min_ms  med_ms  max_ms  equal  dec  ge50  ge100
_op01_000001     calibration/intrinsics     1      -4.5 -708016.5     0.0     0.0     0.0      0    0     0      0
_op01_000001     calibration/extrinsics     1      -4.5 -708016.5     0.0     0.0     0.0      0    0     0      0
_op01_000001                cam:cam_ext 21269      -4.5     -47.5    33.2    33.3    33.4      0    0     0      0
_op01_000001         cam:cam_left_wrist 21239      -1.0     -32.2    26.4    34.0    69.4      0    0    11      0
_op01_000001         imu/cam_left_wrist 21239      -1.0     -32.2    26.4    34.0    69.4      0    0    11      0
_op01_000001        cam:cam_right_wrist 21239      -0.5     -33.6    22.6    34.0    69.9      

## 4 Video

Section 3 of the standard, checked on the temporary streams with ffprobe and PyAV. It also reports frame rates, durations and the out-of-frame share of the hands.

In [5]:
def probe(path):
    cmd = ["ffprobe", "-v", "error", "-count_packets", "-show_streams", "-show_format", "-of", "json", str(path)]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    return json.loads(r.stdout)

def frac(s):
    try:
        a, b = str(s).split("/")
        return float(a) / float(b)
    except Exception:
        return float("nan")

def decode_range(e, cam, start, stop, info=None):
    st = E[e]["chan"]["/cameras/" + cam]
    off = st["offsets"]
    codec = av.CodecContext.create("h264", "r")
    with open(st["stream_path"], "rb") as fh:
        fh.seek(int(off[start]))
        for i in range(start, stop):
            data = fh.read(int(off[i + 1] - off[i]))
            for fr in codec.decode(av.Packet(data)):
                yield fr
        for fr in codec.decode(None):
            yield fr
    if info is not None:
        info["has_b_frames"] = getattr(codec, "has_b_frames", None)

def snap(st, i):
    # move a start index back to the previous idr
    k = np.searchsorted(st["idr"], i, side="right") - 1
    return int(st["idr"][k]) if k >= 0 else 0

def windows(st):
    n, w = st["n"], 900
    if FULL_DECODE or n <= 3 * w:
        return [(0, n)]
    mid = snap(st, max(0, n // 2 - w // 2))
    end = snap(st, max(0, n - w))
    return [(0, w), (mid, mid + w), (end, n)]

def check_window(e, cam, a, b):
    out = dict(frames=0, ident=0, ident_run=0, black=0, black_runs=0, sizes=set(), bpict=0, bframes=None, errors=0)
    prev, run, in_black = None, 0, False
    info = {}
    try:
        for fr in decode_range(e, cam, a, b, info):
            p = fr.planes[0]
            g = np.frombuffer(p, dtype=np.uint8).reshape(p.height, p.line_size)[:, :p.width]
            hsh = hashlib.blake2b(g.tobytes(), digest_size=8).digest()
            if prev is not None and hsh == prev:
                run += 1
                out["ident"] += 1
                out["ident_run"] = max(out["ident_run"], run)
            else:
                run = 0
            prev = hsh
            gs = g[::2, ::2]
            blk = gs.mean() < 16 and gs.std() < 8
            if blk:
                out["black"] += 1
                if not in_black:
                    out["black_runs"] += 1
            in_black = blk
            out["sizes"].add((fr.width, fr.height))
            if "B" in str(getattr(fr.pict_type, "name", fr.pict_type)).upper().split("."):
                out["bpict"] += 1
            out["frames"] += 1
    except Exception as ex:
        out["errors"] += 1
        out["msg"] = str(ex)[:100]
    out["bframes"] = info.get("has_b_frames")
    return out

PROBE, V = {}, {}
for e in EPISODES:
    for cam in EXPECTED_CAMS:
        st = E[e]["chan"].get("/cameras/" + cam)
        if st is None:
            continue
        try:
            js = probe(st["stream_path"])
            s = js["streams"][0]
            PROBE[(e, cam)] = dict(codec=s.get("codec_name"), nb=len(js["streams"]), nb_fmt=js.get("format", {}).get("nb_streams"),
                                   w=s.get("width"), h=s.get("height"), pix=s.get("pix_fmt"), range=s.get("color_range"),
                                   matrix=s.get("color_space"), profile=s.get("profile"), level=s.get("level"),
                                   packets=int(s.get("nb_read_packets", -1)), rfr=s.get("r_frame_rate"), afr=s.get("avg_frame_rate"))
        except Exception as ex:
            PROBE[(e, cam)] = None
            print("ffprobe failed", e[-12:], cam, ex)
        agg = dict(frames=0, ident=0, ident_run=0, black=0, black_runs=0, sizes=set(), bpict=0, bframes=set(), errors=0, fed=0)
        for a, b in tqdm(windows(st), desc=f"decode {e[-8:]} {cam}", leave=False):
            r = check_window(e, cam, a, b)
            for k in ("frames", "ident", "black", "black_runs", "bpict", "errors"):
                agg[k] += r[k]
            agg["ident_run"] = max(agg["ident_run"], r["ident_run"])
            agg["sizes"] |= r["sizes"]
            agg["bframes"].add(r["bframes"])
            agg["fed"] += b - a
            if r["errors"]:
                print("  decode problem", e[-12:], cam, r.get("msg"))
        V[(e, cam)] = agg

# per camera table
t_rows = []
for (e, cam), pr in PROBE.items():
    st, v = E[e]["chan"]["/cameras/" + cam], V[(e, cam)]
    fps_log = (st["n"] - 1) / ((st["last"] - st["first"]) / 1e9)
    t_rows.append(dict(ep=e[-8:], cam=cam, size=f"{pr['w']}x{pr['h']}" if pr else "?", pix=pr and pr["pix"], rng=pr and pr["range"],
                       matrix=pr and pr["matrix"], prof=pr and f"{pr['profile']} {pr['level']}", pkts=pr and pr["packets"], msgs=st["n"],
                       fps_log=round(fps_log, 3), fps_stream=pr and pr["rfr"], dec=v["frames"], ident=v["ident"], black=v["black"]))
show(pd.DataFrame(t_rows))

def per_cam(fn):
    return [x for x in (fn(e, c) for (e, c) in PROBE) if x]

# S3.1 stream structure
p1 = per_cam(lambda e, c: (PROBE[(e, c)] is None or PROBE[(e, c)]["codec"] != "h264" or PROBE[(e, c)]["nb"] != 1 or
                           PROBE[(e, c)]["packets"] != E[e]["chan"]["/cameras/" + c]["n"]) and f"{e[-8:]}/{c}")
add("S3.1", "3", "codec h264, one stream, packet count equals message count", f"{len(PROBE) - len(p1)} of {len(PROBE)} streams ok" + (f"; bad: {p1[:4]}" if p1 else ""),
    "FLAG" if p1 else "PASS", "re-encode or re-export the streams listed")

# S3.2 colour
p2 = per_cam(lambda e, c: (PROBE[(e, c)] is None or PROBE[(e, c)]["pix"] != "yuv420p" or PROBE[(e, c)]["range"] != "tv" or
                           PROBE[(e, c)]["matrix"] != "bt709") and f"{e[-8:]}/{c} {PROBE[(e, c)] and (PROBE[(e, c)]['pix'], PROBE[(e, c)]['range'], PROBE[(e, c)]['matrix'])}")
profs = sorted({str((p["profile"], p["level"])) for p in PROBE.values() if p})
add("S3.2", "3", "pix_fmt yuv420p, limited range, bt709", f"profile/level {profs}" + (f"; differs: {p2[:3]}" if p2 else "; all ok"),
    "FLAG" if p2 else "PASS", "write the colour range and matrix into the stream (vui) or re-encode as limited range bt709")

# S3.3 nal structure and gop
p3 = []
for e in EPISODES:
    for cam in EXPECTED_CAMS:
        st = E[e]["chan"].get("/cameras/" + cam)
        if st is None:
            continue
        exp = np.arange(0, st["n"], 30)
        gop_ok = np.array_equal(st["idr"], exp)
        if st["no_aud"] or st["idr_no_ps"] or not gop_ok:
            p3.append(f"{e[-8:]}/{cam} no_aud {st['no_aud']} idr_without_sps_pps {st['idr_no_ps']} idr {len(st['idr'])}/{len(exp)} gop30 {gop_ok}")
add("S3.3", "3", "AUD on every access unit, SPS and PPS before each IDR, IDR every 30 units", "ok" if not p3 else "; ".join(p3[:3]),
    "FLAG" if p3 else "PASS", "re-mux with AUD, SPS and PPS before each IDR and keyframe interval 30")

# S3.4 decode
p4 = []
for (e, cam), v in V.items():
    pr = PROBE[(e, cam)]
    want = (pr["w"], pr["h"]) if pr else None
    if (v["bframes"] - {0, None}) or v["bpict"] or v["errors"] or v["frames"] != v["fed"] or v["sizes"] != ({want} if want else v["sizes"]):
        p4.append(f"{e[-8:]}/{cam} has_b_frames {sorted(map(str, v['bframes']))} b_pict {v['bpict']} errors {v['errors']} decoded {v['frames']}/{v['fed']} sizes {sorted(v['sizes'])}")
n_dec = sum(v["frames"] for v in V.values())
add("S3.4", "3", "no B-frames, frame size as in the stream, every unit decodes", f"{n_dec} frames decoded" + ("" if not p4 else "; " + "; ".join(p4[:3])),
    "FLAG" if p4 else "PASS", "re-encode without B-frames; check the streams listed")

# S3.5 frozen frames
fz = [f"{e[-8:]}/{c} decoded identical {v['ident']} (run {v['ident_run']}), identical units {E[e]['chan']['/cameras/' + c]['dup']}" for (e, c), v in V.items()
      if v["ident"] or E[e]["chan"]["/cameras/" + c]["dup"]]
add("S3.5", "3", "no frozen runs of identical frames", "none found" if not fz else "; ".join(fz[:4]), "FLAG" if fz else "PASS",
    "remove or re-record the frozen runs listed")

# S3.6 black runs
bl = [f"{e[-8:]}/{c} {v['black']} frames in {v['black_runs']} runs" for (e, c), v in V.items() if v["black"]]
add("S3.6", "3", "no black runs (mean below 16, std below 8)", "none found" if not bl else "; ".join(bl[:4]), "FLAG" if bl else "PASS",
    "check the black frames listed")

# S3.7 frame rate
FPSLOG = {}
for e in EPISODES:
    for cam in EXPECTED_CAMS:
        st = E[e]["chan"].get("/cameras/" + cam)
        if st is not None:
            FPSLOG[(e, cam)] = (st["n"] - 1) / ((st["last"] - st["first"]) / 1e9)
fps_meta = {e: tofloat(PM[e].get("fps")) for e in EPISODES}
body_bad = [f"{e[-8:]}/{c} {f:.3f}" for (e, c), f in FPSLOG.items() if c in BODY_CAMS and abs(f - fps_meta[e]) > 0.05]
ext = [f for (e, c), f in FPSLOG.items() if c == "cam_ext"]
ext_diff = [f for (e, c), f in FPSLOG.items() if c == "cam_ext" and abs(f - fps_meta[e]) > 0.01]
body_f = [f for (e, c), f in FPSLOG.items() if c in BODY_CAMS]
add("S3.7", "3", "exact fps stated, no mixed fps", f"metadata fps {sorted(set(fps_meta.values()))}, body mean {min(body_f):.3f}-{max(body_f):.3f}, cam_ext {min(ext):.3f}-{max(ext):.3f}" if ext else f"body mean {min(body_f):.3f}-{max(body_f):.3f}",
    "FLAG" if body_bad else ("INFO" if ext_diff else "PASS"), f"body cameras differ from the metadata fps: {body_bad[:3]}")

# S3.8 duration
dd = []
for e in EPISODES:
    for c in BODY_CAMS:
        st = E[e]["chan"].get("/cameras/" + c)
        if st is None:
            continue
        d_est = (st["last"] - st["first"]) / 1e9 + 1 / fps_meta[e]
        if not abs(d_est - DUR[e]) <= 0.1:
            dd.append(f"{e[-8:]}/{c} {d_est:.3f} vs {DUR[e]:.3f}")
dx = [f"{(E[e]['chan']['/cameras/cam_ext']['last'] - E[e]['chan']['/cameras/cam_ext']['first']) / 1e9:.2f}s" for e in EPISODES if "/cameras/cam_ext" in E[e]["chan"]]
add("S3.8", "3", "body camera duration matches duration_sec within 0.1 s", "ok" + (f", cam_ext spans {dx}" if dx else "") if not dd else "; ".join(dd[:4]),
    "FLAG" if dd else "PASS", "make duration_sec equal to last minus first plus one frame period")

# S3.9 resolution
sz_body = {(PROBE[(e, c)]["w"], PROBE[(e, c)]["h"]) for (e, c) in PROBE if c in BODY_CAMS and PROBE[(e, c)]}
sz_ext = {(PROBE[(e, c)]["w"], PROBE[(e, c)]["h"]) for (e, c) in PROBE if c == "cam_ext" and PROBE[(e, c)]}
body_ok = sz_body == {(1920, 1080)}
add("S3.9", "3", "body cameras 1920x1080 and uniform (cam_ext portrait is informational)", f"body {sorted(sz_body)}, cam_ext {sorted(sz_ext)}",
    "FLAG" if not body_ok else ("INFO" if sz_ext and sz_ext != {(1920, 1080)} else "PASS"), "body cameras must be one resolution, 1920x1080")

# S3.10 out of frame
oo = []
for e in EPISODES:
    for hand, top in (("L", HAND_TOPICS[0]), ("R", HAND_TOPICS[1])):
        h = E[e]["hands"].get(top)
        if h is not None:
            oo.append(f"{e[-8:]}{hand} {100 * h['oof'].mean():.1f}%")
add("S3.10", "3", "percentage of out-of-frame hands is reported", "; ".join(oo[:8]), "INFO")

      ep             cam      size     pix rng matrix    prof  pkts  msgs  fps_log fps_stream  dec  ident  black
1_000001        cam_high 1920x1080 yuv420p  tv  bt709 High 41 21240 21240   29.999       60/1 2700      0      0
1_000001  cam_left_wrist 1920x1080 yuv420p  tv  bt709 High 41 21239 21239   29.998       60/1 2729      0      0
1_000001 cam_right_wrist 1920x1080 yuv420p  tv  bt709 High 41 21239 21239   29.998       60/1 2729      0      0
1_000001         cam_ext 1080x1920 yuv420p  tv  bt709 High 42 21269 21269   30.041      120/1 2729      0      0
1_000002        cam_high 1920x1080 yuv420p  tv  bt709 High 41  9390  9390   30.001       60/1 2700      0      0
1_000002  cam_left_wrist 1920x1080 yuv420p  tv  bt709 High 41  9390  9390   30.001       60/1 2700      0      0
1_000002 cam_right_wrist 1920x1080 yuv420p  tv  bt709 High 41  9390  9390   30.001       60/1 2700      0      0
1_000002         cam_ext 1080x1920 yuv420p  tv  bt709 High 42  9403  9403   30.041      120/1 27

## 5 Pose

Section 4 on the hand messages: counts, joint layout, nulls and confidence, and a recomputed coverage that is compared with the metadata.

In [6]:
def leaves(obj, path=()):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from leaves(v, path + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from leaves(v, path + (str(i),))
    else:
        yield path, obj

def find_leaf(obj, hand, pat):
    for path, v in leaves(obj):
        s = "/".join(path).lower()
        if hand in s and re.search(pat, s):
            return "/".join(path), v
    return None

def close_pct(val, want, tol=0.05):
    try:
        v = float(val)
    except Exception:
        return False
    return abs(v - want) <= tol or abs(v * 100 - want) <= tol

HP = {}
probs = {k: [] for k in ("count", "layout", "nullconf", "range", "cov", "flag", "mod")}
always_all, share_all, cov_lines = {}, [], []
for e in EPISODES:
    n_high = E[e]["chan"]["/cameras/cam_high"]["n"]
    P = PM[e]
    cov = P.get("coverage")
    for hand, top in (("left", HAND_TOPICS[0]), ("right", HAND_TOPICS[1])):
        h = E[e]["hands"].get(top)
        if h is None:
            probs["count"].append(f"{e[-8:]} {hand} missing")
            continue
        X, Y, C = h["X"], h["Y"], h["C"]
        null = np.isnan(X) | np.isnan(Y)
        n = len(X)
        if n != n_high:
            probs["count"].append(f"{e[-8:]} {hand} {n} vs {n_high}")
        if h["len_bad"] or h["names_bad"] or len(h["names"] or []) != 25:
            probs["layout"].append(f"{e[-8:]} {hand} len_bad {h['len_bad']} names_bad {h['names_bad']}")
        bad_conf = int((~np.isfinite(C) | (C < 0) | (C > 1)).sum())
        mism = int((null != (C == 0)).sum())
        if mism or bad_conf:
            probs["nullconf"].append(f"{e[-8:]} {hand} null/conf0 mismatch {mism}, confidence outside [0,1] {bad_conf}")
        ok = ~null
        out_xy = int(((X[ok] < 0) | (X[ok] > 1) | (Y[ok] < 0) | (Y[ok] > 1)).sum())
        if out_xy:
            probs["range"].append(f"{e[-8:]} {hand} {out_xy} values outside [0,1]")
        always = [h["names"][k] for k in np.flatnonzero(null.all(axis=0))] if h["names"] else []
        always_all[(e, hand)] = always
        # tracked: at least 15 joints with confidence above 0 and mean of those above 0 at least 0.5
        npos = (C > 0).sum(axis=1)
        meanc = np.where(C > 0, C, 0).sum(axis=1) / np.maximum(npos, 1)
        tracked = (npos >= 15) & (meanc >= 0.5)
        t_pct, d_pct, o_pct = 100 * tracked.mean(), 100 * h["det"].mean(), 100 * h["oof"].mean()
        # hand level confidence
        hi = np.where(ok, C, -1).max(axis=1)
        lo = np.where(ok, C, 2).min(axis=1)
        has = ok.any(axis=1)
        share = float(((hi - lo)[has] < 1e-6).mean()) if has.any() else float("nan")
        share_all.append(share)
        low = t_pct < COVERAGE_FLOOR_PCT
        HP[(e, hand)] = dict(tracked=t_pct, detected=d_pct, oof=o_pct, low=low, always=always, share=share, n=n, tracked_flag_pct=100 * h["trk"].mean())
        # compare with the metadata coverage
        for name, pat, want in (("tracked", "track", t_pct), ("detected", "detect", d_pct), ("out_of_frame", "out", o_pct)):
            f = find_leaf(cov, hand, pat) if cov is not None else None
            if f is None:
                probs["cov"].append(f"{e[-8:]} {hand} no {name} value in coverage")
            elif not close_pct(f[1], want):
                probs["cov"].append(f"{e[-8:]} {hand} {name} metadata {f[1]} vs recomputed {want:.3f}")
        f = find_leaf(cov, hand, "low|flag|floor") if cov is not None else None
        if f is None:
            probs["flag"].append(f"{e[-8:]} {hand} no low_coverage_flag in coverage")
        elif bool(f[1]) != bool(low):
            probs["flag"].append(f"{e[-8:]} {hand} metadata {f[1]} vs recomputed {bool(low)}")
        cov_lines.append(f"{e[-6:]}{hand[0].upper()} trk {t_pct:.1f} det {d_pct:.1f} oof {o_pct:.1f}")
        # modality and tag
        tags_ok = "hand_2d_normalized" in str(E[e]["meta"].get("episode_tags", ""))
        pos_ok = set(h["tag"]) == {"hand_2d_normalized"}
        if P.get("pose_modality") != "2d_normalized" or not tags_ok or not pos_ok:
            probs["mod"].append(f"{e[-8:]} {hand} pose_modality {P.get('pose_modality')}, tag in episode_tags {tags_ok}, position_tag {dict(h['tag'])}")
        idx_ok = np.array_equal(np.array(h["idx"], dtype=float), np.arange(n)) if None not in h["idx"] else False
        if not idx_ok:
            print("INFO sample_index is not 0..n-1 for", e[-12:], hand)

print("\n".join(cov_lines[:8]))
fixes = dict(count="write one hand message per cam_high frame for both hands", layout="use the same 25 joints in the same order in every message",
             nullconf="position must be null exactly where confidence is 0, confidence in [0,1]", range="clip or flag x and y outside [0,1]",
             cov="rewrite coverage in the metadata from the hand messages", flag="rewrite low_coverage_flag with the floor", mod="set pose_modality, episode_tags and position_tag to the 2D normalised values")
rows5 = [("S4.1", "4", "hand message count equals the cam_high frame count, both hands", "count"),
         ("S4.2", "4", "25 joints in the same order in every message", "layout"),
         ("S4.3", "2", "position null exactly where confidence is 0; confidence and x,y inside [0,1]", None),
         ("S4.5", "4", "tracked, detected and out-of-frame coverage equal the metadata (0.05 points)", "cov"),
         ("S4.6", "4", f"low_coverage_flag equals the recomputed flag (floor {COVERAGE_FLOOR_PCT}%)", "flag"),
         ("S4.7", "2", "pose_modality 2d_normalized, tag hand_2d_normalized in tags and messages", "mod")]
for tid, cl, hyp, key in rows5:
    pl = probs["nullconf"] + probs["range"] if key is None else probs[key]
    add(tid, cl, hyp, "ok" if not pl else "; ".join(pl[:3]), "FLAG" if pl else "PASS", fixes.get(key or "nullconf"))

al = sorted({n for v in always_all.values() for n in v})
add("S4.4", "4", "joints that are always null are reported", f"always null: {al}; per episode/hand counts {[len(v) for v in always_all.values()][:8]}", "INFO" if al else "PASS")
shs = [s for s in share_all if s == s]
add("S4.8", "4", "confidence is hand-level (one value per frame)", f"share of frames with one shared value: min {min(shs):.3f}, max {max(shs):.3f}" if shs else "no frames", "INFO")

000001L trk 98.7 det 98.8 oof 0.1
000001R trk 97.4 det 97.5 oof 0.2
000002L trk 99.5 det 99.5 oof 1.9
000002R trk 98.5 det 98.5 oof 0.0
000003L trk 99.9 det 99.9 oof 0.0
000003R trk 97.8 det 97.8 oof 0.0
000004L trk 99.6 det 99.6 oof 0.0
000004R trk 98.1 det 98.1 oof 0.0
S4.1  PASS  hand message count equals the cam_high frame count, both hands | ok
S4.2  PASS  25 joints in the same order in every message | ok
S4.3  PASS  position null exactly where confidence is 0; confidence and x, | ok
S4.5  FLAG  tracked, detected and out-of-frame coverage equal the metadata | 1_000001 left no tracked value in coverage; 1_000001 right no tracked value in coverage; 1_000002 left no tracked value 
S4.6  FLAG  low_coverage_flag equals the recomputed flag (floor 30%) | 1_000001 left no low_coverage_flag in coverage; 1_000001 right no low_coverage_flag in coverage; 1_000002 left no low_co
S4.7  PASS  pose_modality 2d_normalized, tag hand_2d_normalized in tags an | ok
S4.4  INFO  joints that are always n

## 6 Calibration

Reads the two calibration messages and lists, per camera, whether intrinsics and extrinsics are present.

In [7]:
K_NAMES = {"k", "camera_matrix", "intrinsic_matrix", "intrinsics", "matrix"}
D_NAMES = {"d", "dist", "dist_coeffs", "distortion", "distortion_coefficients", "distortion_coeffs"}

def has_key(obj, names):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower() in names and v not in (None, [], {}, ""):
                return True
            if has_key(v, names):
                return True
    elif isinstance(obj, list):
        return any(has_key(v, names) for v in obj)
    return False

def status_texts(obj):
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if "status" in str(k).lower() and isinstance(v, str):
                out.append(v)
            out += status_texts(v)
    elif isinstance(obj, list):
        for v in obj:
            out += status_texts(v)
    return out

def find_cam(obj, cam, path=()):
    if isinstance(obj, dict):
        if path and path[-1] == cam or any(obj.get(k) == cam for k in ("name", "camera", "frame_id", "camera_name")):
            return obj
        for k, v in obj.items():
            r = find_cam(v, cam, path + (str(k),))
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_cam(v, cam, path)
            if r is not None:
                return r
    return None

def leaf_dicts(obj):
    if isinstance(obj, dict):
        kids = [v for v in obj.values() if isinstance(v, (dict, list))]
        if not any(isinstance(v, dict) for v in obj.values()):
            yield obj
        for v in kids:
            yield from leaf_dicts(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from leaf_dicts(v)

CALIB = dict(lack_k=[], lack_d=[], status=Counter(), n_unavail=0, n_tf=0)
for e in EPISODES:
    cal = E[e]["calib"]
    intr = (cal.get(CALIB_TOPICS[0]) or [None])[0]
    extr = (cal.get(CALIB_TOPICS[1]) or [None])[0]
    if intr is None or extr is None:
        print("calibration message missing in", e[-12:])
    print(e[-12:], "intrinsics:", json.dumps(intr)[:230])
    for cam in EXPECTED_CAMS:
        d = find_cam(intr, cam) if intr is not None else None
        hk = d is not None and has_key(d, K_NAMES)
        hd = d is not None and has_key(d, D_NAMES)
        stx = status_texts(d) if d is not None else ["camera not found in message"]
        CALIB["status"].update(stx[:1] or ["no status text"])
        if not hk:
            CALIB["lack_k"].append(f"{e[-6:]}/{cam}")
        if not hd:
            CALIB["lack_d"].append(f"{e[-6:]}/{cam}")
        if e == EPISODES[0]:
            print(f"  {cam}: K {hk}, D {hd}, status {stx[:1]}")
    # transforms
    print(e[-12:], "extrinsics:", json.dumps(extr)[:230])
    tf = []
    for d in (leaf_dicts(extr) if extr is not None else []):
        st_t = status_texts(d)
        vals = [v for k, v in d.items() if not re.search(r"status|reason|name|note|frame|source|id|from|to", str(k).lower())]
        if st_t or any(re.search(r"to_world|to_rig|transform|extrinsic", str(k).lower()) for k in d):
            tf.append((st_t[0] if st_t else "no status", any(v not in (None, [], {}, "") for v in vals) and not any("unavail" in s.lower() for s in st_t)))
    if extr is not None and not tf:
        for path, v in leaves(extr):
            if re.search(r"to_world|to_rig|transform|extrinsic", "/".join(path).lower()):
                tf.append(("null value" if v is None else "value", v is not None))
    CALIB["n_tf"] += len(tf)
    CALIB["n_unavail"] += sum(1 for s, a in tf if not a)
    CALIB["status"].update(s for s, _ in tf)

print("status texts:", dict(CALIB["status"]))
bad = CALIB["lack_k"] or CALIB["n_unavail"] or CALIB["n_tf"] == 0
add("S4.9", "4.6", "camera calibration present (K, distortion, camera to world, camera to rig)",
    f"cameras without K: {len(CALIB['lack_k'])}, without distortion: {len(CALIB['lack_d'])}, transforms {CALIB['n_tf']} of which unavailable {CALIB['n_unavail']}; status {dict(CALIB['status'])}",
    "FLAG" if bad else "PASS", "declared deviation: provide K, distortion and both extrinsics or keep the declaration in the Vendor Spec")

_op01_000001 intrinsics: {"timestamp": {"sec": 1791729902, "nsec": 489455111}, "calibration_id": "rig01_2026-10-11", "cameras": [{"camera_name": "cam_high", "frame_id": "CamHigh", "image_frame_id": "CamHighImage", "width": 1920, "height": 1080, "distortio
  cam_high: K True, D True, status []
  cam_left_wrist: K True, D True, status []
  cam_right_wrist: K True, D True, status []
  cam_ext: K True, D True, status []
_op01_000001 extrinsics: {"timestamp": {"sec": 1791729902, "nsec": 489455111}, "calibration_id": "rig01_2026-10-11", "transforms": [{"name": "Q_RigCamHigh", "reference": "Rig", "target": "CamHigh", "translation_m": [0.0, 0.0, 0.0], "quaternion_xyzw": [0.0
_op01_000002 intrinsics: {"timestamp": {"sec": 1791730662, "nsec": 487532889}, "calibration_id": "rig01_2026-10-11", "cameras": [{"camera_name": "cam_high", "frame_id": "CamHigh", "image_frame_id": "CamHighImage", "width": 1920, "height": 1080, "distortio
_op01_000002 extrinsics: {"timestamp": {"sec": 1791730662, "nsec": 4

## 7 Labels

Checks the instruction and the spans in the metadata and prints simple statistics of the labels.

In [8]:
def pick(s, keys):
    for k in keys:
        if k in s:
            return s[k]
    return None

def get_spans(sp):
    out = []
    if isinstance(sp, dict):
        sp = sp.get("spans", list(sp.values()))
    if not isinstance(sp, list):
        return out
    for i, s in enumerate(sp):
        if isinstance(s, dict):
            lab = pick(s, ["label", "text", "description", "annotation", "instruction"])
            out.append(dict(index=i, start=tofloat(pick(s, ["start", "start_time", "start_sec", "t0", "begin"])),
                            end=tofloat(pick(s, ["end", "end_time", "end_sec", "stop", "t1"])), label="" if lab is None else str(lab)))
        elif isinstance(s, (list, tuple)) and len(s) >= 3:
            out.append(dict(index=i, start=tofloat(s[0]), end=tofloat(s[1]), label=str(s[2])))
    return out

CTRL = re.compile(r"[\x00-\x1f\x7f]")
SPANS = {}
lab_rows, p_instr, p_span, p_empty, p_words = [], [], [], [], []
n_all = n_clean = 0
for e in EPISODES:
    P = PM[e]
    instr = str(P.get("language_instruction", ""))
    nw = len(instr.split())
    if nw < 10:
        p_instr.append(f"{e[-8:]} {nw} words")
    sp = get_spans(P.get("spans"))
    SPANS[e] = sp
    if not sp:
        p_span.append(f"{e[-8:]} no spans parsed from {str(E[e]['meta'].get('spans'))[:80]}")
        continue
    starts = [s["start"] for s in sp]
    if starts != sorted(starts):
        p_span.append(f"{e[-8:]} spans not sorted")
    if abs(sp[0]["start"]) > 1 / 30:
        p_span.append(f"{e[-8:]} first start {sp[0]['start']}")
    if not abs(sp[-1]["end"] - DUR[e]) <= 0.1:
        p_span.append(f"{e[-8:]} last end {sp[-1]['end']} vs duration {DUR[e]}")
    for a, b in zip(sp, sp[1:]):
        if abs(b["start"] - a["end"]) > 1 / 30:
            p_span.append(f"{e[-8:]} span {a['index']}-{b['index']} gap or overlap {b['start'] - a['end']:.3f}s")
    for s in sp:
        lab = s["label"].strip()
        words = len(lab.split())
        issues = []
        if not lab:
            issues.append("empty"); p_empty.append(f"{e[-8:]}#{s['index']} empty")
        if CTRL.search(s["label"]):
            issues.append("control"); p_empty.append(f"{e[-8:]}#{s['index']} control chars")
        if lab and not (2 <= words <= 25):
            issues.append("words"); p_words.append(f"{e[-8:]}#{s['index']} {words} words")
        if lab.lower() == instr.strip().lower():
            issues.append("same as instruction"); p_words.append(f"{e[-8:]}#{s['index']} equals instruction")
        n_all += 1
        n_clean += not issues
    durs = np.array([s["end"] - s["start"] for s in sp])
    labs = [s["label"].strip() for s in sp]
    first = Counter(l.split()[0].lower() for l in labs if l.split())
    lab_rows.append(dict(ep=e[-8:], spans=len(sp), distinct=len(set(labs)), longest_s=round(float(durs.max()), 1), median_s=round(float(np.median(durs)), 1),
                         top_first_words=", ".join(f"{w} {c}" for w, c in first.most_common(4))))
show(pd.DataFrame(lab_rows))
pct_clean = 100 * n_clean / n_all if n_all else float("nan")
print(f"percent clean: {pct_clean:.1f} ({n_clean} of {n_all} spans)")

add("S4.10", "4", "language_instruction has at least 10 words", "ok" if not p_instr else "; ".join(p_instr), "FLAG" if p_instr else "PASS",
    "write an instruction of at least 10 words")
add("S4.11", "5", "spans sorted, start at 0, end at duration, no gap or overlap beyond 1/30 s", "ok" if not p_span else "; ".join(p_span[:3]),
    "FLAG" if p_span else "PASS", "make the spans consecutive over the full episode")
add("S4.12", "4", "every label non-empty, no control characters, 2-25 words, differs from the instruction",
    f"{pct_clean:.1f}% clean" + ("" if not (p_empty or p_words) else "; " + "; ".join((p_empty + p_words)[:4])),
    "FLAG" if p_empty else ("INFO" if p_words else "PASS"), "fill the empty labels and remove control characters")
add("S4.13", "4", "label statistics are reported", f"{n_all} spans, distinct labels {sum(r['distinct'] for r in lab_rows)}, longest span {max([r['longest_s'] for r in lab_rows] or [0])}s", "INFO")

      ep  spans  distinct  longest_s  median_s                   top_first_words
1_000001     18         9      192.0      12.0   chop 9, wash 5, pick 3, reach 1
1_000002     23        14       51.0       9.0      stir 8, pick 4, add 4, put 2
1_000003     18        14       66.0      15.0   stir 5, tear 4, reach 2, pour 2
1_000004     25        14       80.0      20.0 stir 8, adjust 3, pour 3, cover 3
percent clean: 100.0 (84 of 84 spans)
S4.10 PASS  language_instruction has at least 10 words | ok
S4.11 PASS  spans sorted, start at 0, end at duration, no gap or overlap b | ok
S4.12 PASS  every label non-empty, no control characters, 2-25 words, diff | 100.0% clean
S4.13 INFO  label statistics are reported | 84 spans, distinct labels 51, longest span 192.0s


## 8 Metadata

Section 5 of the standard. Every key of the metadata record is checked and the placeholders are listed.

In [9]:
RX_ID = re.compile(r"^[A-Za-z0-9]+_\d{4}-\d{2}-\d{2}_[a-z0-9]+_\d{6}$")
RX_ISO = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(\.\d+)?Z$")

def flat_strings(obj):
    out = []
    for path, v in leaves(obj):
        if isinstance(v, str):
            out.append(("/".join(path), v))
    return out

q = {k: [] for k in ("keys", "id", "iso", "fps", "steps", "rig", "cam", "ids", "tags", "prov")}
PH, PH_IN = [], []
ids_seen = Counter(E[e]["meta"].get("episode_id") for e in EPISODES)
for e in EPISODES:
    m, P = E[e]["meta"], PM[e]
    t = e[-8:]
    miss = [k for k in REQUIRED_KEYS if k not in m or not isinstance(m[k], str)]
    if miss:
        q["keys"].append(f"{t} {miss}")
    eid = m.get("episode_id", "")
    if not RX_ID.match(eid) or ids_seen[eid] != 1:
        q["id"].append(f"{t} {eid!r} unique {ids_seen[eid] == 1}")
    s, p = m.get("capture_start_iso", ""), m.get("capture_stop_iso", "")
    try:
        ok_fmt = bool(RX_ISO.match(s) and RX_ISO.match(p)) and int(s[:4]) > 2020 and int(p[:4]) > 2020
        d_ns = iso_ns(p) - iso_ns(s)
        if not ok_fmt or d_ns <= 0 or abs(iso_ns(s) + DUR[e] * 1e9 - iso_ns(p)) > 1e6:
            q["iso"].append(f"{t} {s} .. {p} duration {DUR[e]}")
    except Exception as ex:
        q["iso"].append(f"{t} cannot parse {s!r} {p!r}")
    fv = tofloat(P.get("fps"))
    if not (fv == fv and fv > 0):
        q["fps"].append(f"{t} fps {m.get('fps')!r}")
    n_high = E[e]["chan"]["/cameras/cam_high"]["n"]
    if str(P.get("max_timesteps")).strip() != str(n_high) and tofloat(P.get("max_timesteps")) != n_high:
        q["steps"].append(f"{t} max_timesteps {P.get('max_timesteps')} vs {n_high}")
    rig = P.get("rig")
    desc = rig.strip() if isinstance(rig, str) else next((str(v) for k, v in (rig or {}).items() if "descr" in str(k).lower() and v), "") if isinstance(rig, dict) else ""
    if not desc:
        q["rig"].append(f"{t} rig has no description: {str(rig)[:60]}")
    cams = P.get("cameras")
    if isinstance(cams, list):
        cmap = {str(c.get("name") or c.get("camera") or c.get("id")): c for c in cams if isinstance(c, dict)}
    else:
        cmap = cams if isinstance(cams, dict) else {}
    miss_c = [c for c in EXPECTED_CAMS if c not in cmap]
    if miss_c:
        q["cam"].append(f"{t} missing cameras {miss_c}")
    for c in EXPECTED_CAMS:
        info = cmap.get(c)
        if not isinstance(info, dict):
            continue
        lost = [k for k in ("role", "resolution", "is_pose_source") if k not in info]
        if lost:
            q["cam"].append(f"{t}/{c} missing {lost}")
        nums = re.findall(r"\d+", json.dumps(info.get("resolution")))
        pr = PROBE.get((e, c))
        if len(nums) >= 2 and pr and (int(nums[0]), int(nums[1])) != (pr["w"], pr["h"]):
            q["cam"].append(f"{t}/{c} resolution {nums[:2]} vs stream {pr['w']}x{pr['h']}")
    n_pose = sum(1 for c in cmap.values() if isinstance(c, dict) and c.get("is_pose_source") in (True, "true", "True"))
    if n_pose != 1:
        q["cam"].append(f"{t} is_pose_source true for {n_pose} cameras")
    for k in ("user_id", "task_id", "task_name"):
        if not str(m.get(k, "")).strip():
            q["ids"].append(f"{t} {k} empty")
    tn = str(m.get("task_name", ""))
    words = [w for w in re.split(r"[\s_\-]+", tn) if w]
    if len(words) < 2 or sum(ch.isalpha() for ch in tn) < 0.7 * max(len(tn), 1):
        q["ids"].append(f"{t} task_name looks like a code: {tn!r}")
    tags = P.get("episode_tags")
    tl = [str(x) for x in tags] if isinstance(tags, list) else [f"{k}:{v}" for k, v in tags.items()] if isinstance(tags, dict) else [str(tags)]
    has_v = any("vendor" in x.lower() for x in tl)
    has_f = any("format" in x.lower() for x in tl)
    n_mod = sum(1 for x in tl if re.search(r"modality|rgb|hand|imu|video|monocular", x.lower()))
    if not (has_v and has_f and n_mod):
        q["tags"].append(f"{t} vendor {has_v} source_format {has_f} modality tags {n_mod}")
    prov = P.get("source_provenance")
    lv = list(leaves(prov)) if prov is not None else []
    found = {}
    for name, pat in (("site", "site"), ("operator", "operator"), ("session_or_clip", "session|clip")):
        hit = next(((("/".join(pp)), v) for pp, v in lv if re.search(pat, "/".join(pp).lower()) and str(v).strip()), None)
        found[name] = hit
        if hit is None:
            q["prov"].append(f"{t} no {name}")
    if e == EPISODES[0]:
        print("provenance found:", {k: (v[1] if v else None) for k, v in found.items()})
        print("tags:", tl[:8])
    # placeholders
    for k, v in m.items():
        for ph in PLACEHOLDERS:
            if v.strip() == ph:
                PH.append(f"{t}.{k}={ph}")
            elif ph in v and not v.strip().startswith(("{", "[")):
                PH_IN.append(f"{t}.{k} contains {ph}")
    for k in JSONISH:
        for path, v in flat_strings(P.get(k)):
            if v.strip() in PLACEHOLDERS:
                PH.append(f"{t}.{k}/{path}={v.strip()}")

fx = dict(keys="write all 17 keys as strings", id="episode_id must match <site>_<date>_<x>_<6 digits> and be unique", iso="use ISO-8601 UTC with Z, start plus duration equals stop",
          fps="write fps as an exact number", steps="max_timesteps must equal the cam_high frame count", rig="add a rig description", cam="complete the cameras map",
          ids="fill user_id, task_id and a human-readable task_name", tags="add vendor, source_format and modality tags", prov="add site, operator and a session or clip id")
names5 = [("S5.1", "keys", "all 17 keys present and strings"), ("S5.2", "id", "episode_id format and unique"),
          ("S5.3", "iso", "ISO-8601 UTC after 2020, stop after start, start plus duration equals stop (1 ms)"),
          ("S5.4", "fps", "fps is an exact number"), ("S5.5", "steps", "max_timesteps equals the cam_high message count"),
          ("S5.6", "rig", "rig has a description"), ("S5.7", "cam", "cameras map: four names, role, resolution, one pose source"),
          ("S5.8", "ids", "user_id, task_id and a task_name made of words"), ("S5.9", "tags", "episode_tags with vendor, source_format and modality"),
          ("S5.10", "prov", "source_provenance has site, operator and session or clip id")]
for tid, k, hyp in names5:
    add(tid, "5", hyp, "ok" if not q[k] else "; ".join(q[k][:3]), "FLAG" if q[k] else "PASS", fx[k])
add("S5.11", "5", "no placeholder values left", f"equal to a placeholder: {len(PH)} e.g. {PH[:4]}" + (f"; also inside text: {PH_IN[:3]}" if PH_IN else ""),
    "FLAG" if PH else "PASS", "replace vendorX, op01 and site-unknown before delivery (also where they are part of episode_id)")

provenance found: {'site': 'site-unknown', 'operator': 'op01', 'session_or_clip': '20261011_200844'}
tags: ['vendorX', 'egocentric', 'hand_pose', 'hand_2d_normalized', 'source_format:mp4_h264_imu_txt', 'imu']
S5.1  PASS  all 17 keys present and strings | ok
S5.2  PASS  episode_id format and unique | ok
S5.3  FLAG  ISO-8601 UTC after 2020, stop after start, start plus duration | 1_000001 2026-10-11T14:45:02.494Z .. 2026-10-11T14:56:50.506Z duration 708.0; 1_000002 2026-10-11T14:57:42.512Z .. 2026-
S5.4  PASS  fps is an exact number | ok
S5.5  PASS  max_timesteps equals the cam_high message count | ok
S5.6  PASS  rig has a description | ok
S5.7  FLAG  cameras map: four names, role, resolution, one pose source | 1_000001 missing cameras ['cam_ext']; 1_000002 missing cameras ['cam_ext']; 1_000003 missing cameras ['cam_ext'] | fix: 
S5.8  PASS  user_id, task_id and a task_name made of words | ok
S5.9  PASS  episode_tags with vendor, source_format and modality | ok
S5.10 PASS  source_provena

## 9 Volume

Section 6: bitrate per camera and gigabytes per hour, compared with the reference.

In [10]:
VOL, vol_rows, GBH = {}, [], {}
bits_cam, secs_cam = Counter(), Counter()
tot_v, tot_f, tot_s = 0, 0, 0.0
groups = Counter()
for e in EPISODES:
    d = DUR[e] if DUR[e] > 0 else (E[e]["chan"]["/cameras/cam_high"]["last"] - E[e]["chan"]["/cameras/cam_high"]["first"]) / 1e9
    vb = 0
    row = dict(ep=e[-8:], dur_s=round(d, 1))
    for c in EXPECTED_CAMS:
        st = E[e]["chan"].get("/cameras/" + c)
        if st is None:
            continue
        b = int(st["au"].sum())
        VOL[(e, c)] = b * 8 / d / 1e6
        row[c] = round(VOL[(e, c)], 2)
        bits_cam[c] += b * 8
        secs_cam[c] += d
        vb += b
    GBH[e] = vb / d * 3600 / 1e9
    row["GB_per_h"] = round(GBH[e], 2)
    vol_rows.append(row)
    tot_v += vb
    tot_f += PATHS[e].stat().st_size
    tot_s += d
    for top, st in E[e]["chan"].items():
        g = "video" if top.startswith("/cameras/") else "imu" if "/imu/" in top else "hand" if "/hand/" in top else "calibration" if "/calibration/" in top else "other"
        groups[g] += st["bytes"]
show(pd.DataFrame(vol_rows))
mean_cam = {c: bits_cam[c] / secs_cam[c] / 1e6 for c in EXPECTED_CAMS if secs_cam[c]}
GBH_ALL = tot_v / tot_s * 3600 / 1e9
GBH_FILE = tot_f / tot_s * 3600 / 1e9
print("mean Mbps per camera:", {c: round(v, 2) for c, v in mean_cam.items()})
print(f"whole delivery: video {GBH_ALL:.2f} GB/h, files {GBH_FILE:.2f} GB/h, {tot_f / 1e9:.2f} GB in {tot_s / 60:.1f} min")
print("bytes per group (MB):", {g: round(v / 1e6, 1) for g, v in groups.items()})

ABOVE = sorted({c for (e, c), v in VOL.items() if v > BITRATE_REF[1]})
BELOW = sorted({c for (e, c), v in VOL.items() if v < BITRATE_REF[0]})
mx = max(VOL.values()) if VOL else float("nan")
add("S6.1", "6", f"bitrate per camera inside {BITRATE_REF[0]}-{BITRATE_REF[1]} Mbps",
    f"mean {({c: round(v, 2) for c, v in mean_cam.items()})}, max {mx:.2f}; above {BITRATE_REF[1]}: {ABOVE}; below {BITRATE_REF[0]}: {BELOW}",
    "FLAG" if ABOVE else "INFO", "justification required in the Vendor Spec for " + ", ".join(ABOVE))
add("S6.2", "6", "delivery volume in GB per hour (high end about 14 GB/h needs a reason)", f"video {GBH_ALL:.2f} GB/h, files {GBH_FILE:.2f} GB/h", "INFO")

      ep  dur_s  cam_high  cam_left_wrist  cam_right_wrist  cam_ext  GB_per_h
1_000001  708.0     11.77            6.71             4.81     1.60     11.20
1_000002  313.0     11.59            5.73             4.39     1.52     10.46
1_000003  347.0     11.58            6.27             4.60     1.57     10.81
1_000004  600.0     11.70            6.15             4.63     1.57     10.82
mean Mbps per camera: {'cam_high': 11.69, 'cam_left_wrist': 6.31, 'cam_right_wrist': 4.65, 'cam_ext': 1.57}
whole delivery: video 10.90 GB/h, files 11.67 GB/h, 6.38 GB in 32.8 min
bytes per group (MB): {'calibration': 0.0, 'video': 5966.6, 'imu': 44.7, 'hand': 343.1}
S6.1  FLAG  bitrate per camera inside 5-8 Mbps | mean {'cam_high': 11.69, 'cam_left_wrist': 6.31, 'cam_right_wrist': 4.65, 'cam_ext': 1.57}, max 11.77; above 8: ['cam_hi
S6.2  INFO  delivery volume in GB per hour (high end about 14 GB/h needs a | video 10.90 GB/h, files 11.67 GB/h


## 10 Review sheets, pose

Picks frames from cam_high, draws the delivered 2D joints on them and saves pages for the manual check. The review template is written once and never overwritten.

In [11]:
POSE_DIR = REPORTS / "13_pose_sheets"
POSE_TPL = REPORTS / "13_pose_review_template.csv"
LEFT_C, RIGHT_C = (0, 200, 0), (255, 0, 255)
FONT = cv2.FONT_HERSHEY_SIMPLEX

def streams_ready():
    return all(E[e]["chan"][t]["stream_path"].exists() for e in EPISODES for t in VIDEO_TOPICS if t in E[e]["chan"])

def get_frame(e, cam, idx):
    st = E[e]["chan"]["/cameras/" + cam]
    start = snap(st, idx)
    for j, fr in enumerate(decode_range(e, cam, start, idx + 1)):
        if start + j == idx:
            return fr.to_ndarray(format="bgr24")
    return None

def pick_pose_frames(e, rng):
    n = E[e]["chan"]["/cameras/cam_high"]["n"]
    nb = 10
    bins = [range(int(n * b / nb), int(n * (b + 1) / nb)) for b in range(nb)]
    chosen = set()
    und = [~E[e]["hands"][t]["det"] for t in HAND_TOPICS if t in E[e]["hands"]]
    if und:
        m = min(len(u) for u in und)
        cand = [int(i) for i in np.flatnonzero(np.logical_or.reduce([u[:m] for u in und])) if i < n]
        chosen.update(rng.sample(cand, min(3, len(cand))))
    k = SAMPLES_POSE_PER_EPISODE
    for b in range(nb):
        quota = k // nb + (1 if b < k % nb else 0)
        have = sum(1 for i in chosen if i in bins[b])
        pool = [i for i in bins[b] if i not in chosen]
        chosen.update(rng.sample(pool, min(max(0, quota - have), len(pool))))
    return sorted(chosen)

def draw_pose(img, e, i):
    H, W = img.shape[:2]
    cv2.rectangle(img, (0, 0), (W, 20), (0, 0, 0), -1)
    t = (E[e]["chan"]["/cameras/cam_high"]["log"][i] - E[e]["chan"]["/cameras/cam_high"]["log"][0]) / 1e9
    cv2.putText(img, f"{e} frame {i} t {t:.2f}s", (4, 14), FONT, 0.42, (255, 255, 255), 1)
    line = 0
    for tag, top, col in (("L", HAND_TOPICS[0], LEFT_C), ("R", HAND_TOPICS[1], RIGHT_C)):
        h = E[e]["hands"].get(top)
        if h is None or i >= len(h["X"]):
            continue
        x, y, c = h["X"][i], h["Y"][i], h["C"][i]
        ok = ~np.isnan(x) & ~np.isnan(y)
        for k in np.flatnonzero(ok):
            cv2.circle(img, (int(x[k] * W), int(y[k] * H)), 3, col, -1)
        if ok.any():
            names = [str(n).lower() for n in h["names"]]
            wk = next((k for k, n in enumerate(names) if "wrist" in n), None)
            if wk is None or not ok[wk]:
                wk = int(np.flatnonzero(ok)[0])
            cv2.putText(img, tag, (int(x[wk] * W) + 5, int(y[wk] * H) - 5), FONT, 0.6, col, 2)
            txt = f"{tag} conf {np.nanmean(c[ok]):.2f}"
        else:
            txt = f"{tag} none det {int(h['det'][i])} oof {int(h['oof'][i])}"
        cv2.putText(img, txt, (6, H - 8 - 18 * line), FONT, 0.5, col, 1)
        line += 1
    return img

have_pages = POSE_DIR.exists() and any(POSE_DIR.glob("page_*.jpg"))
if have_pages and POSE_TPL.exists() and not FORCE:
    print("pose sheets exist, skipping (set FORCE = True to draw them again)")
elif not streams_ready():
    print("temporary streams are missing, run cell 2 again")
else:
    POSE_DIR.mkdir(parents=True, exist_ok=True)
    rng = random.Random(SEED)
    items = [(e, i) for e in EPISODES for i in pick_pose_frames(e, rng)]
    tpl, cells = [], []
    for j, (e, i) in enumerate(tqdm(items, desc="pose frames")):
        img = get_frame(e, "cam_high", i)
        cell = np.zeros((360, 640, 3), np.uint8) if img is None else draw_pose(cv2.resize(img, (640, 360)), e, i)
        cells.append(cell)
        tpl.append(dict(episode_id=e, frame=i, page=j // 6 + 1, slot=j % 6 + 1, left_ok="", right_ok="", note=""))
        if len(cells) == 6 or j == len(items) - 1:
            page = np.zeros((720, 1920, 3), np.uint8)
            for s, c in enumerate(cells):
                r, col = divmod(s, 3)
                page[r * 360:(r + 1) * 360, col * 640:(col + 1) * 640] = c
            cv2.imwrite(str(POSE_DIR / f"page_{j // 6 + 1:02d}.jpg"), page, [cv2.IMWRITE_JPEG_QUALITY, 90])
            cells = []
    print(len(items), "frames on", (len(items) + 5) // 6, "pages in", POSE_DIR)
    if POSE_TPL.exists():
        print("template exists, not overwritten:", POSE_TPL.name)
    else:
        pd.DataFrame(tpl).to_csv(POSE_TPL, index=False, encoding="utf-8-sig")
        print("wrote", POSE_TPL.name)

pose frames: 100%|██████████| 120/120 [00:32<00:00,  3.69it/s]

120 frames on 20 pages in c:\workstation\3RD YEAR\ego_pipeline\reports\13_pose_sheets
wrote 13_pose_review_template.csv


## 11 Review sheets, labels

Picks spans, decodes three stills per camera for each and saves one picture per span. The temporary streams are deleted at the end of this cell.

In [12]:
LAB_DIR = REPORTS / "13_label_sheets"
LAB_TPL = REPORTS / "13_label_review_template.csv"

def fit(img, w, h):
    s = min(w / img.shape[1], h / img.shape[0])
    r = cv2.resize(img, (max(1, int(img.shape[1] * s)), max(1, int(img.shape[0] * s))))
    out = np.zeros((h, w, 3), np.uint8)
    y0, x0 = (h - r.shape[0]) // 2, (w - r.shape[1]) // 2
    out[y0:y0 + r.shape[0], x0:x0 + r.shape[1]] = r
    return out

def nearest(log, t_ns):
    j = int(np.searchsorted(log, t_ns))
    cand = [k for k in (j - 1, j) if 0 <= k < len(log)]
    return min(cand, key=lambda k: abs(int(log[k]) - t_ns))

def wrap(text, n=115):
    words, lines, cur = text.split(), [], ""
    for w in words:
        if len(cur) + len(w) + 1 > n:
            lines.append(cur)
            cur = w
        else:
            cur = (cur + " " + w).strip()
    return (lines + [cur])[:2]

try:
    have = LAB_DIR.exists() and any(LAB_DIR.glob("*.jpg"))
    if have and LAB_TPL.exists() and not FORCE:
        print("label sheets exist, skipping (set FORCE = True to draw them again)")
    elif not streams_ready():
        print("temporary streams are missing, run cell 2 again")
    else:
        LAB_DIR.mkdir(parents=True, exist_ok=True)
        rng = random.Random(SEED)
        chosen = []
        for e in EPISODES:
            idx = [s["index"] for s in SPANS[e]]
            chosen += [(e, i) for i in rng.sample(idx, min(5, len(idx)))]
        rest = [(e, s["index"]) for e in EPISODES for s in SPANS[e] if (e, s["index"]) not in chosen]
        chosen += rng.sample(rest, max(0, min(SAMPLES_LABELS - len(chosen), len(rest))))
        chosen = sorted(chosen, key=lambda x: (EPISODES.index(x[0]), x[1]))
        tpl = []
        for e, i in tqdm(chosen, desc="label spans"):
            s = SPANS[e][i]
            sheet = np.zeros((90 + 800, 1440, 3), np.uint8)
            cv2.putText(sheet, f"{e}  span {i}  {s['start']:.2f}-{s['end']:.2f}s", (8, 22), FONT, 0.6, (255, 255, 255), 1)
            for k, ln in enumerate(wrap(s["label"] or "(empty label)")):
                cv2.putText(sheet, ln, (8, 48 + 24 * k), FONT, 0.6, (120, 220, 255), 1)
            for r, cam in enumerate(("cam_ext", "cam_high")):
                st = E[e]["chan"]["/cameras/" + cam]
                for c, p in enumerate((0.25, 0.5, 0.75)):
                    t_ns = START[e] + int((s["start"] + p * (s["end"] - s["start"])) * 1e9)
                    j = nearest(st["log"], t_ns)
                    img = get_frame(e, cam, j)
                    cell = np.zeros((400, 480, 3), np.uint8) if img is None else fit(img, 480, 400)
                    cv2.putText(cell, f"{cam} {int(p * 100)}% f{j}", (6, 18), FONT, 0.5, (0, 255, 255), 1)
                    sheet[90 + r * 400:90 + (r + 1) * 400, c * 480:(c + 1) * 480] = cell
            cv2.imwrite(str(LAB_DIR / f"{e}_span_{i:03d}.jpg"), sheet, [cv2.IMWRITE_JPEG_QUALITY, 88])
            tpl.append(dict(episode_id=e, span_index=i, start_time=round(s["start"], 3), end_time=round(s["end"], 3), label=s["label"], correct="", note=""))
        print(len(chosen), "span sheets in", LAB_DIR)
        if LAB_TPL.exists():
            print("template exists, not overwritten:", LAB_TPL.name)
        else:
            pd.DataFrame(tpl).to_csv(LAB_TPL, index=False, encoding="utf-8-sig")
            print("wrote", LAB_TPL.name)
finally:
    n_del = 0
    for f in INTERIM.glob("*.h264"):
        f.unlink()
        n_del += 1
    print("deleted", n_del, "temporary streams")

label spans: 100%|██████████| 30/30 [00:19<00:00,  1.54it/s]


30 span sheets in c:\workstation\3RD YEAR\ego_pipeline\reports\13_label_sheets
wrote 13_label_review_template.csv
deleted 16 temporary streams


## 12 Pose review

Reads the filled pose review file if it exists and computes the share of correct hands with a Wilson interval.

In [13]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def read_review(path):
    for enc in ("utf-8-sig", "cp1252"):
        try:
            return pd.read_csv(path, dtype=str, sep=None, engine="python", encoding=enc).fillna("")
        except UnicodeDecodeError:
            continue
    raise ValueError("cannot read " + str(path))

POSE_REVIEW = None
f12 = REPORTS / "13_pose_review.csv"
if f12.exists():
    rv = read_review(f12)
    for c in ("left_ok", "right_ok"):
        rv[c] = rv[c].str.strip().str.lower()
    res = {}
    for hand, c in (("left", "left_ok"), ("right", "right_ok")):
        y, n = int((rv[c] == "y").sum()), int((rv[c] == "n").sum())
        res[hand] = (y, n)
    ty, tn = sum(v[0] for v in res.values()), sum(v[1] for v in res.values())
    used = int(((rv.left_ok.isin(["y", "n"])) | (rv.right_ok.isin(["y", "n"]))).sum())
    if ty + tn:
        txt = []
        for name, (y, n) in list(res.items()) + [("overall", (ty, tn))]:
            lo, hi = wilson(y, y + n)
            txt.append(f"{name} {100 * y / max(y + n, 1):.1f}% ({y}/{y + n}, 95% interval {100 * lo:.1f}-{100 * hi:.1f})")
        POSE_REVIEW = dict(rows=used, overall=100 * ty / (ty + tn), n=ty + tn, lo=100 * wilson(ty, ty + tn)[0], hi=100 * wilson(ty, ty + tn)[1], hands=res)
        print("rows used:", used)
        print("\n".join(txt))
        add("S4.14", "4", "hand pose is correct in the reviewed frames (Wilson 95% interval)", "; ".join(txt) + f"; rows used {used}", "INFO")
    else:
        print("review file has no y or n values yet")
        add("S4.14", "4", "hand pose is correct in the reviewed frames", "pose review is pending (no y or n values)", "SKIP")
else:
    print("pose review is pending: fill 13_pose_review_template.csv and save a copy as 13_pose_review.csv")
    add("S4.14", "4", "hand pose is correct in the reviewed frames", "pose review is pending", "SKIP")

pose review is pending: fill 13_pose_review_template.csv and save a copy as 13_pose_review.csv
S4.14 SKIP  hand pose is correct in the reviewed frames | pose review is pending


## 13 Label review

Same for the label review file. The interval is reported because a small sample cannot prove 99 percent.

In [14]:
LABEL_REVIEW = None
f13 = REPORTS / "13_label_review.csv"
if f13.exists():
    rl = read_review(f13)
    rl["correct"] = rl["correct"].str.strip().str.lower()
    y, n = int((rl.correct == "y").sum()), int((rl.correct == "n").sum())
    if y + n:
        lo, hi = wilson(y, y + n)
        LABEL_REVIEW = dict(n=y + n, correct=100 * y / (y + n), lo=100 * lo, hi=100 * hi, wrong=n)
        print(f"rows used: {y + n}, correct {100 * y / (y + n):.1f}% (95% interval {100 * lo:.1f}-{100 * hi:.1f})")
        bad = rl[rl.correct == "n"]
        for r in bad.head(50).itertuples():
            print("  n:", r.episode_id[-8:], r.span_index, r.label)
        print("A sample of this size cannot prove 99 percent accuracy, so the interval is reported instead.")
        res = "PASS" if lo >= 0.99 else ("FLAG" if hi < 0.99 else "INFO")
        add("S4.15", "4", "labels at least 99 percent accurate (Wilson 95% interval)", f"{100 * y / (y + n):.1f}% of {y + n}, interval {100 * lo:.1f}-{100 * hi:.1f}, wrong {n}",
            res, "correct the labels marked n and review more spans")
    else:
        print("label review file has no y or n values yet")
        add("S4.15", "4", "labels at least 99 percent accurate", "label review is pending (no y or n values)", "SKIP")
else:
    print("label review is pending: fill 13_label_review_template.csv and save a copy as 13_label_review.csv")
    add("S4.15", "4", "labels at least 99 percent accurate", "label review is pending", "SKIP")

label review is pending: fill 13_label_review_template.csv and save a copy as 13_label_review.csv
S4.15 SKIP  labels at least 99 percent accurate | label review is pending


## 14 mcap command line tool

Finds the mcap tool and runs info and doctor on every final file. The output is saved in reports\13_mcap_cli.txt.

In [15]:
def find_mcap():
    cands = []
    if MCAP_EXE:
        cands.append(Path(MCAP_EXE))
    w = shutil.which("mcap")
    if w:
        cands.append(Path(w))
    cands += [Path(sys.executable).parent / "mcap.exe", Path(sys.executable).parent / "mcap", PROJECT_ROOT / "tools" / "mcap.exe"]
    return next((c for c in cands if c.exists()), None)

def run_cli(args):
    try:
        r = subprocess.run(args, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=1800)
        return r.returncode, r.stdout, r.stderr
    except subprocess.TimeoutExpired:
        return "timeout", "", "timed out after 30 minutes"
    except Exception as ex:
        return "error", "", str(ex)

exe = find_mcap()
DOCTOR = {}
if exe is None:
    print("mcap tool not found. Set MCAP_EXE to the path that: where.exe mcap shows")
    add("V1", "2", "mcap doctor returns 0 for every final file", "mcap tool not found", "INFO")
    add("V2", "2", "mcap doctor --strict-message-order returns 0", "mcap tool not found", "INFO")
else:
    print("using", exe)
    out = []
    rc, so, se = run_cli([str(exe), "--help"])
    out.append(f"$ mcap --help\nreturn code {rc}\n{so}\n{se}\n")
    strict = {}
    for e in EPISODES:
        for name, args in (("info", ["info"]), ("doctor", ["doctor"]), ("strict", ["doctor", "--strict-message-order"])):
            rc, so, se = run_cli([str(exe)] + args + [str(PATHS[e])])
            out.append(f"$ mcap {' '.join(args)} {PATHS[e].name}\nreturn code {rc}\n--- stdout\n{so}\n--- stderr\n{se}\n")
            if name == "doctor":
                DOCTOR[e] = rc
            if name == "strict":
                strict[e] = rc
            print(f"  {e[-12:]} {name}: {rc}")
    (REPORTS / "13_mcap_cli.txt").write_text("\n".join(out), encoding="utf-8")
    bad = [e[-12:] for e, rc in DOCTOR.items() if rc != 0]
    add("V1", "2", "mcap doctor returns 0 for every final file", f"{len(DOCTOR) - len(bad)} of {len(DOCTOR)} return 0" + (f"; non-zero: {bad}" if bad else ""),
        "FLAG" if bad else "PASS", "read reports/13_mcap_cli.txt and fix the errors doctor reports")
    sb = [e[-12:] for e, rc in strict.items() if rc != 0]
    add("V2", "2", "mcap doctor --strict-message-order returns 0", f"non-zero: {sb}" if sb else "all return 0", "INFO")

using c:\workstation\3RD YEAR\ego_pipeline\tools\mcap.exe
  _op01_000001 info: 0
  _op01_000001 doctor: 0
  _op01_000001 strict: 0
  _op01_000002 info: 0
  _op01_000002 doctor: 0
  _op01_000002 strict: 0
  _op01_000003 info: 0
  _op01_000003 doctor: 0
  _op01_000003 strict: 0
  _op01_000004 info: 0
  _op01_000004 doctor: 0
  _op01_000004 strict: 0
V1    PASS  mcap doctor returns 0 for every final file | 4 of 4 return 0
V2    INFO  mcap doctor --strict-message-order returns 0 | all return 0


## 15 Deviation register

Builds the list of deviations from the measured results. Status is OPEN, DECLARED or CLOSED.

In [16]:
REG = []
def reg(i, clause, topic, finding, status, sentence):
    REG.append(dict(id=i, clause=clause, topic=topic, finding=finding, status=status, vendor_spec_sentence=sentence))

# calibration
reg("D01", "4.6", "camera calibration",
    f"cameras without K {len(CALIB['lack_k'])}, without distortion {len(CALIB['lack_d'])}, transforms unavailable {CALIB['n_unavail']} of {CALIB['n_tf']}; status {dict(CALIB['status'])}",
    "DECLARED" if (CALIB["lack_k"] or CALIB["n_unavail"] or CALIB["n_tf"] == 0) else "CLOSED",
    "Camera calibration (intrinsics, distortion, extrinsics) is not available for this delivery and is written as null with a status text.")

# 2d pose
mods = sorted({str(PM[e].get("pose_modality")) for e in EPISODES})
reg("D02", "2/5", "2D pose instead of metric 3D", f"pose_modality values {mods}, position_tag {sorted({str(t) for e in EPISODES for h in E[e]['hands'].values() for t in h['tag']})}",
    "DECLARED" if mods == ["2d_normalized"] else "CLOSED",
    "Hand pose is 2D image-normalised on cam_high and tagged hand_2d_normalized; there is no metric 3D pose.")

# hand level confidence
shs = [v["share"] for v in HP.values() if v["share"] == v["share"]]
hl = bool(shs) and min(shs) >= 0.99
reg("D03", "2", "hand-level confidence", f"share of frames with one confidence value for all joints: {min(shs):.3f} to {max(shs):.3f}" if shs else "no hand frames",
    "DECLARED" if hl else "CLOSED", "Confidence is one value per hand and frame, copied to every joint, not an independent value per joint.")

# always missing joints
al = sorted({n for v in HP.values() for n in v["always"]})
reg("D04", "2", "joints that are always missing", f"{len(al)} joints always null: {al}", "DECLARED" if al else "CLOSED",
    f"{len(al)} of the 25 joints are always null with confidence 0 because the tracker does not output them.")

# layout
names0 = next((h["names"] for e in EPISODES for h in E[e]["hands"].values()), [])
same = all(h["names_bad"] == 0 for e in EPISODES for h in E[e]["hands"].values())
reg("D05", "2", "provisional 25-joint layout", f"{len(names0)} joint names, same order in every message: {same}; first names {names0[:4]}", "DECLARED",
    "The 25-joint layout and joint order follow the files and are provisional until the client confirms its layout.")

# extra topics
extra = sorted({t for e in EPISODES for t in E[e]["chan"]} - set(STANDARD_TOPICS))
reg("D06", "2", "non-standard IMU and calibration topics", f"{len(extra)} extra topics: {extra}", "DECLARED" if extra else "CLOSED",
    "The IMU channels and the two calibration channels are extra topics outside the standard topic list.")

# cam_ext
ex_sz = sorted({f"{PROBE[k]['w']}x{PROBE[k]['h']}" for k in PROBE if k[1] == "cam_ext" and PROBE[k]})
ex_fps = [FPSLOG[k] for k in FPSLOG if k[1] == "cam_ext"]
ex_late = sum(E[e]["chan"]["/cameras/cam_ext"]["n_50"] for e in EPISODES if "/cameras/cam_ext" in E[e]["chan"])
reg("D07", "3", "cam_ext portrait and frame rate", f"size {ex_sz}, mean fps {min(ex_fps):.3f}-{max(ex_fps):.3f} against metadata {sorted(set(fps_meta.values()))}, intervals >= 50 ms {ex_late}" if ex_fps else "no cam_ext",
    "DECLARED", "cam_ext is a fixed phone in portrait orientation with about 30.04 fps and a true timestamp for every frame.")

# bitrate
if ABOVE:
    reg("D08", "6", "bitrate above the reference", f"cameras above {BITRATE_REF[1]} Mbps: {ABOVE}; max {max(VOL.values()):.2f} Mbps; video {GBH_ALL:.2f} GB/h",
        "OPEN", f"Bitrate of {', '.join(ABOVE)} is above the {BITRATE_REF[1]} Mbps reference; justification: <to be written>.")

# length
long_e = [(e[-12:], round(DUR[e], 1)) for e in EPISODES if DUR[e] > TARGET_MAX_S]
if long_e:
    reg("D09", "1", "episode length above the target", f"above {TARGET_MAX_S}s: {long_e}; hard cap {HARD_MAX_S}s", "OPEN" if any(d > HARD_MAX_S for _, d in long_e) else "DECLARED",
        f"{len(long_e)} episodes are longer than the {TARGET_MAX_S} s target and shorter than the {HARD_MAX_S} s hard cap.")

# timestamp nudges from provenance
nudge = []
for e in EPISODES:
    for path, v in leaves(PM[e].get("source_provenance")):
        if "nudge" in "/".join(path).lower():
            try:
                nudge.append(float(v))
            except Exception:
                pass
    prov = PM[e].get("source_provenance")
    if isinstance(prov, dict):
        for k, v in prov.items():
            if "nudge" in str(k).lower() and isinstance(v, (list, dict)):
                nudge.append(float(len(v)))
reg("D10", "2", "timestamp nudges", f"nudge count found in source_provenance: {int(sum(nudge))}" if nudge else "no nudge entry found in source_provenance",
    "DECLARED" if nudge and sum(nudge) else "OPEN" if not nudge else "CLOSED",
    f"{int(sum(nudge))} repeated timestamps were moved by 1 microsecond; this is recorded in source_provenance." if nudge else "Check the provenance for the timestamp nudge count.")

# late frames
late = {c: sum(E[e]["chan"]["/cameras/" + c]["n_50"] for e in EPISODES if "/cameras/" + c in E[e]["chan"]) for c in EXPECTED_CAMS}
late_big = {c: sum(E[e]["chan"]["/cameras/" + c]["n_100"] for e in EPISODES if "/cameras/" + c in E[e]["chan"]) for c in EXPECTED_CAMS}
reg("D11", "3", "late-stamped frames", f"intervals >= 50 ms per camera {late}, >= 100 ms {late_big}", "DECLARED" if sum(late.values()) else "CLOSED",
    f"{sum(late.values())} video intervals are 50 ms or longer; they are kept as recorded.")

# placeholders
reg("D12", "5", "placeholder identity values", f"{len(PH)} values equal a placeholder: {PH[:6]}", "OPEN" if PH else "CLOSED",
    "Identity values (vendor, operator, site) are placeholders and must be replaced before delivery.")

# camera models
pix = sorted({str(p["pix"]) for p in PROBE.values() if p})
cam_keys = sorted({k for e in EPISODES if isinstance(PM[e].get("cameras"), dict) for c in PM[e]["cameras"].values() if isinstance(c, dict) for k in c})
reg("D13", "5", "camera models unknown, RGB monocular", f"pix_fmt {pix}, camera metadata keys {cam_keys}, no depth topic: {not any('depth' in t for t in extra)}", "DECLARED",
    "Camera and lens models are unknown; all cameras are RGB and monocular without depth.")

# accuracy evidence
if POSE_REVIEW:
    reg("D14", "4", "hand pose accuracy evidence", f"{POSE_REVIEW['overall']:.1f}% of {POSE_REVIEW['n']} hand ratings correct, 95% interval {POSE_REVIEW['lo']:.1f}-{POSE_REVIEW['hi']:.1f}", "CLOSED",
        f"Hand pose was reviewed by eye on {POSE_REVIEW['n']} hand ratings: {POSE_REVIEW['overall']:.1f}% correct.")
else:
    reg("D14", "4", "hand pose accuracy evidence", "pose review pending", "OPEN", "Hand pose accuracy is not yet measured.")
if LABEL_REVIEW:
    reg("D15", "4", "label accuracy evidence", f"{LABEL_REVIEW['correct']:.1f}% of {LABEL_REVIEW['n']} spans correct, 95% interval {LABEL_REVIEW['lo']:.1f}-{LABEL_REVIEW['hi']:.1f}",
        "CLOSED" if LABEL_REVIEW["lo"] >= 99 else "OPEN", f"Labels were reviewed on {LABEL_REVIEW['n']} spans: {LABEL_REVIEW['correct']:.1f}% correct; a sample of this size cannot prove 99 percent.")
else:
    reg("D15", "4", "label accuracy evidence", "label review pending", "OPEN", "Label accuracy is not yet measured.")

DEV = pd.DataFrame(REG)
DEV.to_csv(REPORTS / "13_deviation_register.csv", index=False, encoding="utf-8-sig")
print(DEV.status.value_counts().to_dict())
show(DEV[["id", "topic", "status", "finding"]].assign(finding=DEV.finding.str.slice(0, 90)))

{'DECLARED': 11, 'OPEN': 4}
 id                                   topic   status                                                                                    finding
D01                      camera calibration DECLARED cameras without K 0, without distortion 0, transforms unavailable 32 of 36; status {'no st
D02            2D pose instead of metric 3D DECLARED                pose_modality values ['2d_normalized'], position_tag ['hand_2d_normalized']
D03                   hand-level confidence DECLARED                   share of frames with one confidence value for all joints: 1.000 to 1.000
D04          joints that are always missing DECLARED 4 joints always null: ['index_metacarpal', 'little_metacarpal', 'middle_metacarpal', 'ring
D05             provisional 25-joint layout DECLARED 25 joint names, same order in every message: True; first names ['wrist', 'thumb_metacarpal
D06 non-standard IMU and calibration topics DECLARED 5 extra topics: ['/ego/calibration/extrinsics', '/ego/c

## 16 Hypothesis tests

Collects every test of the notebook in one table and saves it.

In [17]:
HT = pd.DataFrame(list(HYP.values()))
order = sorted(range(len(HT)), key=lambda i: (HT.id[i][0], [int(t) for t in re.findall(r"\d+", HT.id[i])]))
HT = HT.iloc[order].reset_index(drop=True)
HT.to_csv(REPORTS / "13_hypothesis_check.csv", index=False, encoding="utf-8-sig")
print(HT.result.value_counts().to_dict())
view = HT if len(HT) <= 50 else pd.concat([HT[HT.result != "PASS"], HT[HT.result == "PASS"]]).head(50)
print(view.assign(hypothesis=view.hypothesis.str.slice(0, 55), measured=view.measured.str.slice(0, 70))[["id", "clause", "result", "hypothesis", "measured"]].to_string(index=False))
if len(HT) > 50:
    print(f"showing 50 of {len(HT)} rows, the csv has all of them")

{'PASS': 31, 'INFO': 10, 'FLAG': 8, 'SKIP': 2}
   id clause result                                              hypothesis                                                               measured
 S1.1      1   INFO                 duration at most 1200s, target 120-600s durations 708.0s, 313.0s, 347.0s, 600.0s; outside target: _op01_000001
 S2.1      2   FLAG    no gaps of 100 ms or more in video and hand channels largest gap video/hand 124.2 ms, channels with gaps >= 100 ms: 3; all 
 S2.6      2   INFO                    extra non-standard topics are listed 5 extra: ['/ego/calibration/extrinsics', '/ego/calibration/intrinsics'
 S3.7      3   INFO                          exact fps stated, no mixed fps    metadata fps [30.0], body mean 29.998-30.002, cam_ext 30.041-30.041
 S3.9      3   INFO body cameras 1920x1080 and uniform (cam_ext portrait is                            body [(1920, 1080)], cam_ext [(1080, 1920)]
S3.10      3   INFO            percentage of out-of-frame hands is repo

## 17 Summary

Writes the summary json and the zip file with all 13_ files.

In [18]:
flags = HT[HT.result == "FLAG"]
summary = dict(
    created=dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    episodes=EPISODES,
    results=HT.result.value_counts().to_dict(),
    flags=flags[["id", "hypothesis"]].to_dict("records"),
    deviations=DEV.status.value_counts().to_dict(),
    duration_s={e: DUR[e] for e in EPISODES},
    mbps_mean_per_camera={c: round(v, 3) for c, v in mean_cam.items()},
    gb_per_hour_video=round(GBH_ALL, 3),
    coverage={f"{e[-12:]}_{h}": dict(tracked=round(v["tracked"], 2), detected=round(v["detected"], 2), out_of_frame=round(v["oof"], 2)) for (e, h), v in HP.items()},
    pose_review=POSE_REVIEW,
    label_review=LABEL_REVIEW,
    mcap_doctor={e: DOCTOR.get(e) for e in EPISODES},
    sha256=SHA,
)
(REPORTS / "13_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

zpath = REPORTS / "share_13.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(REPORTS.glob("13_*")):
        if f.is_file():
            z.write(f, f.name)
    for d in ("13_pose_sheets", "13_label_sheets"):
        for f in sorted((REPORTS / d).glob("*")):
            z.write(f, f"{d}/{f.name}")
print("zip:", zpath, f"{zpath.stat().st_size / 1e6:.1f} MB")

print("\nepisodes:", len(EPISODES), " results:", summary["results"], " deviations:", summary["deviations"])
print("\nFLAG rows:")
for r in flags.itertuples():
    print(" ", r.id, r.hypothesis[:70], "->", r.measured.split("| fix:")[-1].strip()[:90])
if flags.empty:
    print("  none")
print("""
next steps:
1 fill reports\\13_pose_review_template.csv (left_ok, right_ok with y, n or na)
2 fill reports\\13_label_review_template.csv (correct with y or n)
3 save the copies as 13_pose_review.csv and 13_label_review.csv in reports
4 run cells 12 to 17 again""")

zip: c:\workstation\3RD YEAR\ego_pipeline\reports\share_13.zip 19.6 MB

episodes: 4  results: {'PASS': 31, 'INFO': 10, 'FLAG': 8, 'SKIP': 2}  deviations: {'DECLARED': 11, 'OPEN': 4}

FLAG rows:
  S2.1 no gaps of 100 ms or more in video and hand channels -> find the gaps above 100 ms in cam:cam_high, hand/joints/left/position, hand/joints/right/p
  S4.5 tracked, detected and out-of-frame coverage equal the metadata (0.05 p -> rewrite coverage in the metadata from the hand messages
  S4.6 low_coverage_flag equals the recomputed flag (floor 30%) -> rewrite low_coverage_flag with the floor
  S4.9 camera calibration present (K, distortion, camera to world, camera to  -> declared deviation: provide K, distortion and both extrinsics or keep the declaration in t
  S5.3 ISO-8601 UTC after 2020, stop after start, start plus duration equals  -> use ISO-8601 UTC with Z, start plus duration equals stop
  S5.7 cameras map: four names, role, resolution, one pose source -> complete the cameras map
  S